# 01 Data Pull And Audit

Load the raw match dataset, validate the schema, define the target, and document the strict pre-match feature boundary.

In [1]:
from pathlib import Path

import pandas as pd

ARTIFACTS_DIR = Path("../artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


def load_matches_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [col.strip() for col in df.columns]
    df["date"] = pd.to_datetime(df["date"], errors="raise")
    df["patch"] = df["patch"].fillna("").astype(str).str.strip()
    df["event"] = df["event"].fillna("unknown").astype(str).str.strip()
    df["blue_team"] = df["blue_team"].astype(str).str.strip()
    df["red_team"] = df["red_team"].astype(str).str.strip()
    df["winner"] = df["winner"].astype(str).str.strip()
    df["blue_team_win"] = (df["winner"] == df["blue_team"]).astype(int)
    return df.sort_values(["date"], kind="mergesort").reset_index(drop=True)[
        [
            "season",
            "date",
            "event",
            "patch",
            "blue_team",
            "red_team",
            "winner",
            "blue_team_win",
        ]
    ]


DATA_PATH = Path("../data/matchs_stats.csv")
df = load_matches_csv(DATA_PATH)
df.to_csv(ARTIFACTS_DIR / "clean_matches.csv", index=False)
df.head()

,season,date,event,patch,blue_team,red_team,winner,ban_1_blue_team,ban_2_blue_team,ban_3_blue_team,...,jungler_blue_team,mid_blue_team,adc_blue_team,support_blue_team,top_red_team,jungler_red_team,mid_red_team,adc_red_team,support_red_team,blue_team_win
0,1,2011-06-18,Main,<NA>,TSM,Team_gamed_21-de,TSM,Rumble,Nidalee,Sona,...,TheOddOne,Reginald,Chaox,Xpecial,Reyk,Zylor,Kev1n,CandyPanda,Nyph,1
1,1,2011-06-18,Main,<NA>,Counter_Logic_Gaming,Xan,Counter_Logic_Gaming,Mordekaiser,Vayne,Jarvan IV,...,Saintvicious,bigfatlp,Chauster,Elementz,Axion,Radeon6870,Vech,d4rkness,iNtrigueD,1
2,1,2011-06-18,Main,<NA>,TSM,Counter_Logic_Gaming,TSM,Nidalee,Rumble,Nunu,...,TheOddOne,Reginald,Chaox,Xpecial,HotshotGG,Saintvicious,bigfatlp,Chauster,Elementz,1
3,1,2011-06-18,Main,<NA>,Team_gamed_21-de,Xan,Team_gamed_21-de,Mordekaiser,Ashe,Vladimir,...,Zylor,Reyk,CandyPanda,Nyph,Vech,Radeon6870,Axion,d4rkness,iNtrigueD,1
4,1,2011-06-18,Main,<NA>,TSM,Xan,Xan,Corki,Jarvan IV,Twisted Fate,...,TheOddOne,Reginald,Chaox,Xpecial,Vech,Radeon6870,Axion,d4rkness,iNtrigueD,0


In [2]:
df.shape, df["date"].min(), df["date"].max(), df["blue_team_win"].dtype

((1070, 38),
 Timestamp('2011-06-18 00:00:00'),
 Timestamp('2022-11-06 00:00:00'),
 Int64Dtype())

In [3]:
blocked_prefixes = ("ban_", "pick_", "top_", "jungler_", "mid_", "adc_", "support_")
blocked_exact = {"winner", "blue_team_win"}
allowed_columns = [
    col for col in df.columns
    if col not in blocked_exact and not col.startswith(blocked_prefixes)
]
blocked_columns = [col for col in df.columns if col not in allowed_columns]
allowed_columns, blocked_columns[:20]

(['season', 'date', 'event', 'patch', 'blue_team', 'red_team'],
 ['winner',
  'ban_1_blue_team',
  'ban_2_blue_team',
  'ban_3_blue_team',
  'ban_4_blue_team',
  'ban_5_blue_team',
  'ban_1_red_team',
  'ban_2_red_team',
  'ban_3_red_team',
  'ban_4_red_team',
  'ban_5_red_team',
  'pick_1_blue_team',
  'pick_2_blue_team',
  'pick_3_blue_team',
  'pick_4_blue_team',
  'pick_5_blue_team',
  'pick_1_red_team',
  'pick_2_red_team',
  'pick_3_red_team',
  'pick_4_red_team'])

In [4]:
df.isna().mean().sort_values(ascending=False).head(20)

ban_5_blue_team     0.330841
ban_5_red_team      0.328037
ban_4_blue_team     0.327103
ban_4_red_team      0.327103
patch               0.114019
ban_1_blue_team     0.001869
ban_1_red_team      0.001869
red_team            0.000000
event               0.000000
date                0.000000
season              0.000000
blue_team           0.000000
ban_3_blue_team     0.000000
ban_2_blue_team     0.000000
ban_2_red_team      0.000000
winner              0.000000
ban_3_red_team      0.000000
pick_1_blue_team    0.000000
pick_2_blue_team    0.000000
pick_3_blue_team    0.000000
dtype: float64

In [5]:
df["blue_team_win"].value_counts(dropna=False).sort_index()

blue_team_win
0    488
1    582
Name: count, dtype: Int64